# Gravity Student Lab (Lecture 7)

This notebook is designed for students to run gravity regressions with minimal setup.

## Learning objective
- Compare GDP-style gravity without fixed effects to canonical gravity with exporter/importer FE.
- See how FE absorb exporter/importer-only covariates.

## Data
- Prebuilt merged cross-section for 2019 (country-pair level), with expanded covariates.
- No raw Atlas aggregation or CEPII merge steps are required in this notebook.


In [ ]:
# If Colab misses any package, uncomment the next line.
# !pip -q install statsmodels

import io
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

try:
    from IPython.display import display, Markdown
except Exception:
    def display(obj):
        print(obj)

    def Markdown(text):
        return text


In [ ]:
DATA_URL = "https://raw.githubusercontent.com/DTEcon/Teaching_International_Trade_PUBLIC/main/data/gravity_2019_student.csv"
CODEBOOK_URL = "https://raw.githubusercontent.com/DTEcon/Teaching_International_Trade_PUBLIC/main/data/gravity_2019_student_codebook.csv"


def load_student_data() -> pd.DataFrame:
    try:
        data = pd.read_csv(DATA_URL)
        print(f"Loaded data from URL: {DATA_URL}")
        return data
    except Exception as exc:
        print("Could not load dataset from URL.")
        print(f"Reason: {exc}")
        print("Please upload gravity_2019_student.csv from LMS/repo.")
        try:
            from google.colab import files  # type: ignore
            uploaded = files.upload()
            if not uploaded:
                raise RuntimeError("No file uploaded.")
            first_name = next(iter(uploaded))
            data = pd.read_csv(io.BytesIO(uploaded[first_name]))
            print(f"Loaded uploaded file: {first_name}")
            return data
        except Exception as upload_exc:
            raise RuntimeError(
                "Data load failed from both URL and upload path."
            ) from upload_exc


def load_codebook() -> pd.DataFrame | None:
    try:
        return pd.read_csv(CODEBOOK_URL)
    except Exception:
        return None


df_full = load_student_data()
print(f"Rows: {len(df_full):,} | Columns: {len(df_full.columns)}")
print("First columns:", ", ".join(df_full.columns[:12]))

codebook = load_codebook()
if codebook is not None:
    display(Markdown("**Codebook (first rows)**"))
    display(codebook.head(10))


In [ ]:
# @title Gravity Spec Controls
preset_spec = "Canonical FE (OLS)" # @param ["Naive GDP gravity (no FE)", "Cost-proxy gravity (no FE)", "Canonical FE (OLS)", "Canonical FE (PPML)", "Custom"]
estimator = "OLS" # @param ["OLS", "PPML"]
include_fe = True # @param {type:"boolean"}
exporters = "ALL" # @param {type:"string"}
importers = "ALL" # @param {type:"string"}

# @markdown ### Bilateral covariates
cov_ln_dist = True # @param {type:"boolean"}
cov_contig = True # @param {type:"boolean"}
cov_comlang_off = True # @param {type:"boolean"}
cov_rta = True # @param {type:"boolean"}
cov_dist = False # @param {type:"boolean"}
cov_comcol = False # @param {type:"boolean"}
cov_comleg_posttrans = False # @param {type:"boolean"}

# @markdown ### Exporter-only covariates
cov_ln_gdp_o = False # @param {type:"boolean"}
cov_gdp_o = False # @param {type:"boolean"}
cov_ln_pop_o = False # @param {type:"boolean"}
cov_pop_o = False # @param {type:"boolean"}
cov_ln_gdpcap_o = False # @param {type:"boolean"}
cov_gdpcap_o = False # @param {type:"boolean"}
cov_wto_o = False # @param {type:"boolean"}
cov_eu_o = False # @param {type:"boolean"}

# @markdown ### Importer-only covariates
cov_ln_gdp_d = False # @param {type:"boolean"}
cov_gdp_d = False # @param {type:"boolean"}
cov_ln_pop_d = False # @param {type:"boolean"}
cov_pop_d = False # @param {type:"boolean"}
cov_ln_gdpcap_d = False # @param {type:"boolean"}
cov_gdpcap_d = False # @param {type:"boolean"}
cov_wto_d = False # @param {type:"boolean"}
cov_eu_d = False # @param {type:"boolean"}

BILATERAL_COVARIATE_VARS = [
    ("ln_dist", "cov_ln_dist"),
    ("contig", "cov_contig"),
    ("comlang_off", "cov_comlang_off"),
    ("rta", "cov_rta"),
    ("dist", "cov_dist"),
    ("comcol", "cov_comcol"),
    ("comleg_posttrans", "cov_comleg_posttrans"),
]

EXPORTER_COVARIATE_VARS = [
    ("ln_gdp_o", "cov_ln_gdp_o"),
    ("gdp_o", "cov_gdp_o"),
    ("ln_pop_o", "cov_ln_pop_o"),
    ("pop_o", "cov_pop_o"),
    ("ln_gdpcap_o", "cov_ln_gdpcap_o"),
    ("gdpcap_o", "cov_gdpcap_o"),
    ("wto_o", "cov_wto_o"),
    ("eu_o", "cov_eu_o"),
]

IMPORTER_COVARIATE_VARS = [
    ("ln_gdp_d", "cov_ln_gdp_d"),
    ("gdp_d", "cov_gdp_d"),
    ("ln_pop_d", "cov_ln_pop_d"),
    ("pop_d", "cov_pop_d"),
    ("ln_gdpcap_d", "cov_ln_gdpcap_d"),
    ("gdpcap_d", "cov_gdpcap_d"),
    ("wto_d", "cov_wto_d"),
    ("eu_d", "cov_eu_d"),
]

COVARIATE_CHECKBOX_VARS = dict(BILATERAL_COVARIATE_VARS + EXPORTER_COVARIATE_VARS + IMPORTER_COVARIATE_VARS)

PRESETS = {
    "Naive GDP gravity (no FE)": {
        "estimator": "OLS",
        "include_fe": False,
        "covariates": ["ln_dist", "ln_gdp_o", "ln_gdp_d"],
    },
    "Cost-proxy gravity (no FE)": {
        "estimator": "OLS",
        "include_fe": False,
        "covariates": ["ln_dist", "ln_gdp_o", "ln_gdp_d", "contig", "comlang_off", "rta"],
    },
    "Canonical FE (OLS)": {
        "estimator": "OLS",
        "include_fe": True,
        "covariates": ["ln_dist", "contig", "comlang_off", "rta"],
    },
    "Canonical FE (PPML)": {
        "estimator": "PPML",
        "include_fe": True,
        "covariates": ["ln_dist", "contig", "comlang_off", "rta"],
    },
}


def _apply_preset_covariates(covariates: list[str]) -> None:
    selected = set(covariates)
    for covariate, var_name in COVARIATE_CHECKBOX_VARS.items():
        globals()[var_name] = covariate in selected


if preset_spec != "Custom":
    spec = PRESETS[preset_spec]
    estimator = spec["estimator"]
    include_fe = spec["include_fe"]
    _apply_preset_covariates(spec["covariates"])
    control_mode = "Preset mode: estimator, FE, and covariates are auto-applied from preset_spec."
else:
    control_mode = "Custom mode: estimator, FE, and covariates follow your manual control choices."


def get_selected_covariates() -> list[str]:
    return [
        covariate
        for covariate, var_name in COVARIATE_CHECKBOX_VARS.items()
        if bool(globals().get(var_name, False))
    ]


selected_covariates = get_selected_covariates()
if not selected_covariates:
    print("- warning: no covariate selected yet. In Custom mode, tick at least one checkbox before running the regression.")

print("Applied configuration")
print("- preset_spec:", preset_spec)
print("- estimator:", estimator)
print("- include_fe:", include_fe)
print("- selected_covariates:", ", ".join(selected_covariates))
print("- exporters filter:", exporters)
print("- importers filter:", importers)
print("- mode:", control_mode)
print("- note: comcur is not available in Gravity_V202010.dta and is not offered here.")


In [ ]:
EXPORTER_ONLY = {
    "gdp_o",
    "ln_gdp_o",
    "pop_o",
    "ln_pop_o",
    "gdpcap_o",
    "ln_gdpcap_o",
    "wto_o",
    "eu_o",
}
IMPORTER_ONLY = {
    "gdp_d",
    "ln_gdp_d",
    "pop_d",
    "ln_pop_d",
    "gdpcap_d",
    "ln_gdpcap_d",
    "wto_d",
    "eu_d",
}


def parse_iso_list(value: str) -> list[str]:
    value = str(value).strip()
    if value.upper() == "ALL" or value == "":
        return []
    return [item.strip().upper() for item in value.split(",") if item.strip()]


def stars(pvalue: float) -> str:
    if pd.isna(pvalue):
        return ""
    if pvalue < 0.01:
        return "***"
    if pvalue < 0.05:
        return "**"
    if pvalue < 0.1:
        return "*"
    return ""


def apply_fe_absorption_rules(covariate_list: list[str], include_fe: bool) -> tuple[list[str], list[tuple[str, str]]]:
    if not include_fe:
        return list(covariate_list), []

    kept = []
    dropped = []
    for var in covariate_list:
        if var in EXPORTER_ONLY:
            dropped.append((var, "exporter FE"))
        elif var in IMPORTER_ONLY:
            dropped.append((var, "importer FE"))
        else:
            kept.append(var)
    return kept, dropped


def build_formula(dep_var: str, covariate_list: list[str], include_fe: bool) -> str:
    rhs_terms = list(covariate_list)
    if include_fe:
        rhs_terms.append("C(iso3_o)")
        rhs_terms.append("C(iso3_d)")
    if not rhs_terms:
        rhs_terms = ["1"]
    return dep_var + " ~ " + " + ".join(rhs_terms)


def fit_gravity_model(
    data: pd.DataFrame,
    estimator_name: str,
    covariate_list: list[str],
    include_fe: bool,
    exporter_filter: list[str],
    importer_filter: list[str],
):
    work = data.copy()

    if exporter_filter:
        work = work[work["iso3_o"].isin(exporter_filter)].copy()
    if importer_filter:
        work = work[work["iso3_d"].isin(importer_filter)].copy()

    required = ["iso3_o", "iso3_d", "pair_cluster", "trade_value"] + covariate_list
    missing = [col for col in required if col not in work.columns]
    if missing:
        raise ValueError(f"Missing columns in data: {missing}")

    if estimator_name == "OLS":
        work = work[work["trade_value"] > 0].copy()
        work["ln_trade"] = np.log(work["trade_value"])
        dep_var = "ln_trade"
        fit_data = work.dropna(subset=[dep_var] + covariate_list + ["pair_cluster"])
    elif estimator_name == "PPML":
        dep_var = "trade_value"
        fit_data = work.dropna(subset=covariate_list + ["pair_cluster", "trade_value"])
    else:
        raise ValueError("Estimator must be OLS or PPML")

    if fit_data.empty:
        raise ValueError("No rows left after filters and NA handling.")

    formula = build_formula(dep_var, covariate_list, include_fe)

    if estimator_name == "OLS":
        result = smf.ols(formula, data=fit_data).fit(
            cov_type="cluster",
            cov_kwds={"groups": fit_data["pair_cluster"]},
        )
    else:
        result = smf.glm(
            formula,
            data=fit_data,
            family=sm.families.Poisson(),
        ).fit(
            maxiter=200,
            cov_type="cluster",
            cov_kwds={"groups": fit_data["pair_cluster"]},
        )

    return result, fit_data, formula


def build_result_table(result, covariate_list: list[str]) -> pd.DataFrame:
    rows = []
    for var in covariate_list:
        coef = result.params.get(var, np.nan)
        se = result.bse.get(var, np.nan)
        pval = result.pvalues.get(var, np.nan)
        rows.append(
            {
                "variable": var,
                "coef": coef,
                "std_err": se,
                "p_value": pval,
                "stars": stars(pval),
            }
        )

    out = pd.DataFrame(rows, columns=["variable", "coef", "std_err", "p_value", "stars"])
    if out.empty:
        out["coef_with_stars"] = pd.Series(dtype="object")
        out["std_err_fmt"] = pd.Series(dtype="object")
        return out

    out["coef_with_stars"] = out.apply(
        lambda row: "" if pd.isna(row["coef"]) else f"{row['coef']:.4f}{row['stars']}",
        axis=1,
    )
    out["std_err_fmt"] = out["std_err"].apply(lambda x: "" if pd.isna(x) else f"({x:.4f})")
    return out


def absorbed_diagnostics(
    result_table: pd.DataFrame,
    fe_dropped_covariates: list[tuple[str, str]],
) -> list[str]:
    notes = []

    for var, absorber in fe_dropped_covariates:
        notes.append(
            f"{var}: automatically excluded because it is absorbed by {absorber} when include_fe=True."
        )

    dropped = result_table[result_table["coef"].isna()]
    for var in dropped["variable"].tolist():
        notes.append(f"{var}: coefficient not estimated (dropped/collinear or no variation in sample).")

    if not notes:
        notes.append("No obvious FE-absorption/collinearity flags for the estimated covariates.")
    return notes



In [ ]:
selected_covariates = get_selected_covariates()
if not selected_covariates:
    raise ValueError("Please select at least one covariate.")

model_covariates, fe_dropped_covariates = apply_fe_absorption_rules(selected_covariates, include_fe)

selected_exporters = parse_iso_list(exporters)
selected_importers = parse_iso_list(importers)

result, fit_data, formula = fit_gravity_model(
    data=df_full,
    estimator_name=estimator,
    covariate_list=model_covariates,
    include_fe=include_fe,
    exporter_filter=selected_exporters,
    importer_filter=selected_importers,
)

summary_table = build_result_table(result, model_covariates)
notes = absorbed_diagnostics(summary_table, fe_dropped_covariates)

print("Run configuration")
print("- mode:", control_mode)
print("- requested_covariates:", ", ".join(selected_covariates))
print(
    "- estimated_covariates:",
    ", ".join(model_covariates) if model_covariates else "(none; FE-only specification)",
)
if fe_dropped_covariates:
    print("- auto_dropped_with_fe:", ", ".join(var for var, _ in fe_dropped_covariates))
print()
print("Model formula:")
print(formula)
print()
print("Sample diagnostics:")
print(f"- N (estimation sample): {len(fit_data):,}")
print(f"- Number of exporters in sample: {fit_data['iso3_o'].nunique():,}")
print(f"- Number of importers in sample: {fit_data['iso3_d'].nunique():,}")
if estimator == "PPML":
    zero_share = (fit_data["trade_value"] == 0).mean()
    print(f"- Zero-flow share used by PPML: {100 * zero_share:.2f}%")

print()
print("Coefficient table (clustered SE by pair):")
if summary_table.empty:
    print("No slope covariates estimated in this specification.")
else:
    display(summary_table[["variable", "coef_with_stars", "std_err_fmt", "p_value"]])

print("FE / collinearity diagnostics:")
for note in notes:
    print("-", note)



## Suggested student experiments

1. Run **Naive GDP gravity (no FE)**, then **Canonical FE (OLS)**. Compare the distance and GDP coefficients.
2. Switch to **Custom**, select one additional bilateral covariate (for example `comleg_posttrans`), and inspect coefficient changes.
3. Toggle `include_fe` off/on with the same covariates and compare identification/interpretation.
4. Restrict exporters/importers (for example Europe-only ISO3 codes) and compare coefficient changes.
